In [ ]:
import os
import json
import cv2
import numpy as np
from skimage.feature import local_binary_pattern
from tqdm import tqdm
from networkb import MLPLivenessClassifier
from scipy.signal import convolve2d

def extract_lpq_histogram(img, win_size=3, freq=1.0):
    img = np.float64(img)
    r = (win_size - 1) / 2
    x = np.arange(-r, r + 1)[np.newaxis]
    w0 = np.ones_like(x)
    w1 = np.exp(-2 * np.pi * 1j * x * freq / win_size)
    w2 = np.conj(w1)
    q1, q2, q3, q4 = w0.T * w1, w1.T * w0, w1.T * w1, w1.T * w2
    filters = [np.real(q1), np.imag(q1), np.real(q2), np.imag(q2),
               np.real(q3), np.imag(q3), np.real(q4), np.imag(q4)]
    lpq_img = np.zeros(img.shape, dtype=np.uint8)
    for i, f in enumerate(filters):
        response = convolve2d(img, f, mode='same', boundary='symm')
        lpq_img += (response > 0).astype(np.uint8) << i
    hist, _ = np.histogram(lpq_img.ravel(), bins=256, range=(0, 256))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-7)
    return hist

def single_scale_retinex(img, sigma=50):
    img_float = np.float32(img) + 1.0
    blurred = cv2.GaussianBlur(img_float, (0, 0), sigma)
    retinex = np.log10(img_float) - np.log10(blurred)
    return np.uint8(cv2.normalize(retinex, None, 0, 255, cv2.NORM_MINMAX))

def process_cropped_image(image_path, mode):
    img = cv2.imread(image_path)
    if img is None: return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    retinex = single_scale_retinex(gray)

    if mode == 'nn_gradients':
        resized = cv2.resize(retinex, (64, 64))
        gx = cv2.Sobel(resized, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(resized, cv2.CV_64F, 0, 1, ksize=3)
        mag = cv2.magnitude(gx, gy)
        return cv2.normalize(mag, None, 0, 1, cv2.NORM_MINMAX).flatten()

    elif mode == 'nn_raw_gray':
        return (cv2.resize(retinex, (64, 64)).astype(np.float32) / 255.0).flatten()

    elif mode == 'nn_spatial_lbp':
        lbp = local_binary_pattern(cv2.resize(retinex, (64, 64)), P=16, R=2, method='uniform')
        return (lbp.astype(np.float32) / 18.0).flatten()

    elif mode == 'nn_high_freq':
        lap = cv2.Laplacian(cv2.resize(retinex, (64, 64)), cv2.CV_64F)
        return cv2.normalize(lap, None, 0, 1, cv2.NORM_MINMAX).flatten()
    
    elif mode == "nn_lpq":
        return extract_lpq_histogram(cv2.resize(retinex, (64, 64)), win_size=7)
    
class CASIAEvaluator:
    def __init__(self, dataset_path, mode):
        self.dataset_path = dataset_path
        self.mode = mode
        self.cache_dir = "extracted_features"
        os.makedirs(self.cache_dir, exist_ok=True)
        self.classifier = MLPLivenessClassifier()

    def _load_features(self, split):
        cache_path = os.path.join(self.cache_dir, f"{split}_{self.mode}.npz")
        if os.path.exists(cache_path):
            data = np.load(cache_path)
            return data['X'], data['y']

        X, y = [], []
        split_dir = os.path.join(self.dataset_path, split)
        for cat, label in {'live': 1, 'spoof': 0}.items():
            path = os.path.join(split_dir, cat)
            for f in tqdm(os.listdir(path), desc=f"Extracting {split} {cat}"):
                feat = process_cropped_image(os.path.join(path, f), self.mode)
                if feat is not None: X.append(feat); y.append(label)
        
        X, y = np.array(X), np.array(y)
        np.savez_compressed(cache_path, X=X, y=y)
        return X, y

    def evaluate(self):
        X_train, y_train = self._load_features('train')
        os.makedirs("models", exist_ok=True)
        plot_path = f"models/training/loss_{self.mode}.png"
        self.classifier.train(X_train, y_train, plot_path=plot_path)
        self.classifier.save_model(f"models/liveness_{self.mode}.pth")

        X_test, y_test = self._load_features('test')
        preds = self.classifier.predict(X_test)

        apcer = np.sum((preds == 1) & (y_test == 0)) / np.sum(y_test == 0)
        bpcer = np.sum((preds == 0) & (y_test == 1)) / np.sum(y_test == 1)
        acer  = (apcer + bpcer) * 50
        print(f"Mode: {self.mode}\nAPCER: {apcer*100:.2f}%\nBPCER: {bpcer*100:.2f}%\nACER: {acer:.2f}%")

        results_path = "results.json"
        results = json.loads(open(results_path).read()) if os.path.exists(results_path) else {}
        results[self.mode] = {
            "APCER": round(apcer * 100, 2),
            "BPCER": round(bpcer * 100, 2),
            "ACER":  round(acer, 2),
        }
        with open(results_path, "w") as f:
            json.dump(results, f, indent=2)

if __name__ == "__main__":
    #Set your task
    CHOSEN_MODE = 'nn_lpq'
    dataset_path = "./casia-fasd"
    evaluator = CASIAEvaluator(dataset_path, CHOSEN_MODE)
    evaluator.evaluate()

In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from main_nn import process_cropped_image
from networkb import MLPLivenessClassifier, LivenessNet

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_PATH = "./casia-fasd"
MODES = ['nn_gradients', 'nn_raw_gray', 'nn_spatial_lbp', 'nn_high_freq', 'nn_lpq']
NUM_EXPLAIN_SAMPLES = 5
# ─────────────────────────────────────────────────────────────────────────────


def load_sample_images(split, category, n=NUM_EXPLAIN_SAMPLES):
    folder = os.path.join(DATASET_PATH, split, category)
    files = [os.path.join(folder, f) for f in os.listdir(folder)
             if f.lower().endswith(('.jpg', '.png', '.bmp'))]
    return files[:n]


class GradCAMForMLP:
    """
    Grad-CAM adapted for LivenessNet (MLP).

    CNN Grad-CAM:  hooks last conv layer → alpha-weighted feature maps → upsample.
    MLP Grad-CAM:  hooks last hidden layer (network[7], 32 units) → alpha-weighted
                   activations → project back to input space via transposed weight
                   matrices, yielding a (input_dim,) importance vector.
    For the four 64×64 spatial modes this reshapes directly to a 64×64 heatmap.

    LivenessNet.network indices:
        [0] Linear(input→128)  [1] LeakyReLU  [2] Dropout
        [3] Linear(128→64)     [4] LeakyReLU  [5] Dropout
        [6] Linear(64→32)      [7] LeakyReLU  [8] Dropout
        [9] Linear(32→1)
    """

    def __init__(self, model: LivenessNet):
        self.model = model
        self.model.eval()
        self._acts: torch.Tensor | None = None
        self._grads: torch.Tensor | None = None
        self._handles: list = []
        self._register_hooks()

    def _register_hooks(self):
        target = self.model.network[7]   # LeakyReLU after 3rd Linear (output dim=32)

        def fwd_hook(module, inp, out):
            self._acts = out

        def bwd_hook(module, grad_in, grad_out):
            self._grads = grad_out[0]

        self._handles.append(target.register_forward_hook(fwd_hook))
        self._handles.append(target.register_full_backward_hook(bwd_hook))

    def remove_hooks(self):
        for h in self._handles:
            h.remove()
        self._handles.clear()

    def compute(self, x_tensor: torch.Tensor):
        """
        Forward + backward pass.  Returns:
          cam_input  (np.ndarray, input_dim): GradCAM importance projected to input
          saliency   (np.ndarray, input_dim): |∂output/∂input| vanilla saliency
          pred_prob  (float): sigmoid probability of 'live'
        """
        self.model.zero_grad()
        x = x_tensor.clone().float().requires_grad_(True)

        logit = self.model(x)
        pred_prob = torch.sigmoid(logit).item()
        logit.backward()

        # ── Vanilla gradient saliency: |∂output/∂input| ───────────────────
        saliency = x.grad.abs().squeeze(0).detach().cpu().numpy()

        # ── GradCAM at last hidden layer ───────────────────────────────────
        # alpha_k = gradient of output w.r.t. k-th hidden unit (GradCAM weight)
        # For 1-D feature vectors there is no spatial GAP step; alpha = gradient directly.
        alpha = self._grads.squeeze(0).detach().cpu()   # (32,)
        acts  = self._acts.squeeze(0).detach().cpu()    # (32,)
        cam_hidden = torch.relu(alpha * acts)            # (32,) — Grad-CAM formula

        # ── Project cam_hidden back to input space via W^T chain ───────────
        # nn.Linear weight shape: (out_features, in_features)
        # so weight.T has shape:  (in_features, out_features)
        # network[6]: Linear(64→32) → W6.T: (64,32)  applied to (32,) → (64,)
        # network[3]: Linear(128→64)→ W3.T: (128,64) applied to (64,) → (128,)
        # network[0]: Linear(in→128)→ W0.T: (in,128) applied to (128,) → (in,)
        W6_T = self.model.network[6].weight.detach().cpu().T   # (64, 32)
        W3_T = self.model.network[3].weight.detach().cpu().T   # (128, 64)
        W0_T = self.model.network[0].weight.detach().cpu().T   # (input_dim, 128)

        proj = W6_T @ cam_hidden   # (64,)
        proj = W3_T @ proj          # (128,)
        cam_input = (W0_T @ proj).numpy()   # (input_dim,)
        cam_input = np.abs(cam_input)

        return cam_input, saliency, pred_prob


# ── Helpers ──────────────────────────────────────────────────────────────────

def _normalize(arr: np.ndarray) -> np.ndarray:
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo + 1e-8)


def _colorize(heatmap_2d: np.ndarray, orig_img: np.ndarray):
    """Return (jet_heatmap_uint8_rgb, alpha_blend_overlay) from a [0,1] 2D heatmap."""
    h, w = orig_img.shape[:2]
    hm = cv2.resize(heatmap_2d.astype(np.float32), (w, h))
    hm_color = (plt.cm.jet(hm)[:, :, :3] * 255).astype(np.uint8)
    overlay = cv2.addWeighted(orig_img, 0.5, hm_color, 0.5, 0)
    return hm_color, overlay


# ── Per-mode explanation ─────────────────────────────────────────────────────

def explain_mode_gradcam(mode: str):
    print(f"\n{'='*60}")
    print(f"  Grad-CAM XAI — Mode: {mode}")
    print(f"{'='*60}")

    model_path = f"models/liveness_{mode}.pth"
    if not os.path.exists(model_path):
        print(f"  [!] No saved model at {model_path}. Run main_nn.py first.")
        return

    classifier = MLPLivenessClassifier.load_model(model_path)
    gradcam = GradCAMForMLP(classifier.model)
    device = classifier.device

    live_paths  = load_sample_images('test', 'live')
    spoof_paths = load_sample_images('test', 'spoof')
    paths  = live_paths  + spoof_paths
    labels = ['live'] * len(live_paths) + ['spoof'] * len(spoof_paths)

    out_dir = f"xai_output/gradcam/{mode}"
    os.makedirs(out_dir, exist_ok=True)

    for img_path, true_label in zip(paths, labels):
        feat = process_cropped_image(img_path, mode)
        if feat is None:
            continue

        x_tensor = torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(device)
        cam_input, saliency, pred_prob = gradcam.compute(x_tensor)
        pred_label = "live" if pred_prob >= 0.5 else "spoof"
        orig_img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        if mode == 'nn_lpq':
            # LPQ outputs a 256-dim histogram — no spatial reshape possible, use bar charts
            fig, axes = plt.subplots(1, 3, figsize=(18, 4))

            axes[0].imshow(orig_img)
            axes[0].axis('off')
            axes[0].set_title(f"True: {true_label}\nPred: {pred_label} ({pred_prob:.2f})")

            c_colors = plt.cm.jet(_normalize(cam_input))
            axes[1].bar(range(len(cam_input)), cam_input, color=c_colors)
            axes[1].set_title("Grad-CAM importance (LPQ bins)")
            axes[1].set_xlabel("Bin index")
            axes[1].set_ylabel("Importance")

            s_colors = plt.cm.plasma(_normalize(saliency))
            axes[2].bar(range(len(saliency)), saliency, color=s_colors)
            axes[2].set_title("Gradient Saliency (LPQ bins)")
            axes[2].set_xlabel("Bin index")

        else:
            # Spatial modes: 4096 features → 64×64 heatmap
            cam_2d = _normalize(cam_input.reshape(64, 64))
            sal_2d = _normalize(saliency.reshape(64, 64))

            cam_color, cam_overlay = _colorize(cam_2d, orig_img)
            sal_color, sal_overlay = _colorize(sal_2d, orig_img)

            fig, axes = plt.subplots(2, 3, figsize=(16, 9))

            # Row 0: Grad-CAM (last hidden layer → input projection)
            axes[0, 0].imshow(orig_img)
            axes[0, 0].axis('off')
            axes[0, 0].set_title(f"Original\nTrue: {true_label}")

            axes[0, 1].imshow(cam_color)
            axes[0, 1].axis('off')
            axes[0, 1].set_title("Grad-CAM heatmap\n(last hidden layer projected to input)")

            axes[0, 2].imshow(cam_overlay)
            axes[0, 2].axis('off')
            axes[0, 2].set_title(f"Grad-CAM overlay\nPred: {pred_label} ({pred_prob:.2f})")

            # Row 1: Vanilla gradient saliency for comparison
            axes[1, 0].imshow(orig_img)
            axes[1, 0].axis('off')
            axes[1, 0].set_title("Original")

            axes[1, 1].imshow(sal_color)
            axes[1, 1].axis('off')
            axes[1, 1].set_title("Gradient Saliency heatmap\n|∂output / ∂input|")

            axes[1, 2].imshow(sal_overlay)
            axes[1, 2].axis('off')
            axes[1, 2].set_title("Gradient Saliency overlay")

        fig.suptitle(
            f"Grad-CAM vs Gradient Saliency — Mode: {mode} | {os.path.basename(img_path)}",
            fontsize=11
        )
        plt.tight_layout()

        stem = os.path.splitext(os.path.basename(img_path))[0]
        out_name = f"{out_dir}/{true_label}_{stem}.png"
        plt.savefig(out_name, dpi=100, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {out_name}")

    gradcam.remove_hooks()


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    for mode in MODES:
        explain_mode_gradcam(mode)
    print("\nDone. Check xai_output/gradcam/ for heatmaps.")



  Grad-CAM XAI — Mode: nn_gradients
  Saved: xai_output/gradcam/nn_gradients/live_s25v2f188.png
  Saved: xai_output/gradcam/nn_gradients/live_s28v2f174.png
  Saved: xai_output/gradcam/nn_gradients/live_s8v2f13.png
  Saved: xai_output/gradcam/nn_gradients/live_s9v2f158.png
  Saved: xai_output/gradcam/nn_gradients/live_s16v2f165.png
  Saved: xai_output/gradcam/nn_gradients/spoof_s15vHR_3f108.png
  Saved: xai_output/gradcam/nn_gradients/spoof_s17v8f3.png
  Saved: xai_output/gradcam/nn_gradients/spoof_s8v3f7.png
  Saved: xai_output/gradcam/nn_gradients/spoof_s21vHR_1f56.png
  Saved: xai_output/gradcam/nn_gradients/spoof_s29vHR_3f186.png

  Grad-CAM XAI — Mode: nn_raw_gray
  [!] No saved model at models/liveness_nn_raw_gray.pth. Run main_nn.py first.

  Grad-CAM XAI — Mode: nn_spatial_lbp
  [!] No saved model at models/liveness_nn_spatial_lbp.pth. Run main_nn.py first.

  Grad-CAM XAI — Mode: nn_high_freq
  [!] No saved model at models/liveness_nn_high_freq.pth. Run main_nn.py first.

  Gr

In [3]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from lime import lime_tabular
from main_nn import process_cropped_image, CASIAEvaluator
from networkb import MLPLivenessClassifier

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_PATH = "./casia-fasd"
MODES = ['nn_gradients', 'nn_raw_gray', 'nn_spatial_lbp', 'nn_high_freq', 'nn_lpq']
NUM_EXPLAIN_SAMPLES = 5
NUM_LIME_FEATURES = 50   # top features LIME selects per explanation
# ─────────────────────────────────────────────────────────────────────────────


def load_sample_images(split, category, n=NUM_EXPLAIN_SAMPLES):
    folder = os.path.join(DATASET_PATH, split, category)
    files = [os.path.join(folder, f) for f in os.listdir(folder)
             if f.lower().endswith(('.jpg', '.png', '.bmp'))]
    return files[:n]


def make_predict_fn(model, device):
    """Return [[P(spoof), P(live)]] per sample — required by LIME's classification API."""
    def predict(X):
        tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(tensor)
            p_live = torch.sigmoid(logits).cpu().numpy().flatten()
        return np.stack([1.0 - p_live, p_live], axis=1)
    return predict


def lime_weights_to_array(explanation, label_idx, feature_dim):
    """Unpack LIME's sparse [(feature_index, weight)] list into a dense vector."""
    arr = np.zeros(feature_dim)
    for feat_idx, weight in explanation.local_exp[label_idx]:
        arr[feat_idx] = weight
    return arr


def explain_mode(mode):
    print(f"\n{'='*60}")
    print(f"  LIME XAI — Mode: {mode}")
    print(f"{'='*60}")

    model_path = f"models/liveness_{mode}.pth"
    if not os.path.exists(model_path):
        print(f"  [!] No saved model at {model_path}. Run main_nn.py first.")
        return

    classifier = MLPLivenessClassifier.load_model(model_path)
    model = classifier.model
    device = classifier.device
    predict_fn = make_predict_fn(model, device)

    # Load training features to anchor the LIME neighbourhood distribution
    evaluator = CASIAEvaluator(DATASET_PATH, mode)
    X_train, _ = evaluator._load_features('train')
    feature_dim = X_train.shape[1]

    explainer = lime_tabular.LimeTabularExplainer(
        training_data=X_train,
        feature_names=[f"f_{i}" for i in range(feature_dim)],
        class_names=['spoof', 'live'],
        mode='classification',
        discretize_continuous=True,
        random_state=42,
    )

    live_paths  = load_sample_images('test', 'live')
    spoof_paths = load_sample_images('test', 'spoof')
    paths  = live_paths  + spoof_paths
    labels = ['live'] * len(live_paths) + ['spoof'] * len(spoof_paths)

    out_dir = f"xai_output/lime/{mode}"
    os.makedirs(out_dir, exist_ok=True)

    for img_path, true_label in zip(paths, labels):
        feat = process_cropped_image(img_path, mode)
        if feat is None:
            continue

        explanation = explainer.explain_instance(
            feat,
            predict_fn,
            num_features=NUM_LIME_FEATURES,
            num_samples=1000,
            labels=(1,),   # explain class 1 (live)
        )

        pred_prob = predict_fn(feat.reshape(1, -1))[0, 1]
        pred_label = "live" if pred_prob >= 0.5 else "spoof"

        # Dense weight array for class 1 (live); positive = supports live, negative = supports spoof
        lime_arr = lime_weights_to_array(explanation, label_idx=1, feature_dim=feature_dim)

        orig_img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        if mode == 'nn_lpq':
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))

            axes[0].imshow(orig_img)
            axes[0].set_title(f"True: {true_label} | Pred: {pred_label} ({pred_prob:.2f})")
            axes[0].axis('off')

            colors = ['green' if v > 0 else 'red' for v in lime_arr]
            axes[1].bar(range(feature_dim), lime_arr, color=colors)
            axes[1].set_title("LIME weights (LPQ bins)\nGreen=supports live, Red=supports spoof")
            axes[1].set_xlabel("LPQ bin index")
            axes[1].set_ylabel("LIME weight")

        else:
            # 4096-dim → 64×64 signed heatmap
            heatmap = lime_arr.reshape(64, 64)
            abs_max = np.abs(heatmap).max() + 1e-8
            heatmap_norm = heatmap / abs_max   # range [-1, 1]

            # PiYG: green = positive (live signal), purple = negative (spoof signal)
            heatmap_color = (plt.cm.PiYG((heatmap_norm + 1) / 2)[:, :, :3] * 255).astype(np.uint8)

            h, w = orig_img.shape[:2]
            hm_resized = cv2.resize(heatmap_color, (w, h))
            overlay = cv2.addWeighted(orig_img, 0.5, hm_resized, 0.5, 0)

            fig, axes = plt.subplots(1, 3, figsize=(14, 4))

            axes[0].imshow(orig_img)
            axes[0].set_title(f"Original\nTrue: {true_label}")
            axes[0].axis('off')

            axes[1].imshow(heatmap_color)
            axes[1].set_title("LIME Heatmap\nGreen=live signal, Purple=spoof signal")
            axes[1].axis('off')

            axes[2].imshow(overlay)
            axes[2].set_title(f"Overlay\nPred: {pred_label} ({pred_prob:.2f})")
            axes[2].axis('off')

        fig.suptitle(f"LIME XAI — Mode: {mode} | {os.path.basename(img_path)}", fontsize=12)
        plt.tight_layout()

        stem = os.path.splitext(os.path.basename(img_path))[0]
        out_name = f"{out_dir}/{true_label}_{stem}.png"
        plt.savefig(out_name, dpi=100, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {out_name}")


if __name__ == "__main__":
    for mode in MODES:
        explain_mode(mode)
    print("\nDone. Check xai_output/lime/ for explanations.")



  LIME XAI — Mode: nn_gradients
  Saved: xai_output/lime/nn_gradients/live_s25v2f188.png
  Saved: xai_output/lime/nn_gradients/live_s28v2f174.png
  Saved: xai_output/lime/nn_gradients/live_s8v2f13.png
  Saved: xai_output/lime/nn_gradients/live_s9v2f158.png
  Saved: xai_output/lime/nn_gradients/live_s16v2f165.png
  Saved: xai_output/lime/nn_gradients/spoof_s15vHR_3f108.png
  Saved: xai_output/lime/nn_gradients/spoof_s17v8f3.png
  Saved: xai_output/lime/nn_gradients/spoof_s8v3f7.png
  Saved: xai_output/lime/nn_gradients/spoof_s21vHR_1f56.png
  Saved: xai_output/lime/nn_gradients/spoof_s29vHR_3f186.png

  LIME XAI — Mode: nn_raw_gray
  [!] No saved model at models/liveness_nn_raw_gray.pth. Run main_nn.py first.

  LIME XAI — Mode: nn_spatial_lbp
  [!] No saved model at models/liveness_nn_spatial_lbp.pth. Run main_nn.py first.

  LIME XAI — Mode: nn_high_freq
  [!] No saved model at models/liveness_nn_high_freq.pth. Run main_nn.py first.

  LIME XAI — Mode: nn_lpq
  [!] No saved model at 

In [4]:
import os
import cv2
import numpy as np
import shap
import matplotlib.pyplot as plt
import torch
from main_nn import process_cropped_image, CASIAEvaluator
from networkb import MLPLivenessClassifier, LivenessNet

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATASET_PATH = "./casia-fasd"
MODES = ['nn_gradients', 'nn_raw_gray', 'nn_spatial_lbp', 'nn_high_freq', 'nn_lpq']
NUM_EXPLAIN_SAMPLES = 5   # how many live + spoof images to explain per mode
# ────────────────────────────────────────────────────────────────────────────

def load_sample_images(split, category, n=NUM_EXPLAIN_SAMPLES):
    """Return n image paths from dataset_path/split/category."""
    folder = os.path.join(DATASET_PATH, split, category)
    files = [os.path.join(folder, f) for f in os.listdir(folder)
             if f.lower().endswith(('.jpg', '.png', '.bmp'))]
    return files[:n]

def make_predict_fn(model, device):
    """Wrap the PyTorch model so SHAP can call it like a function."""
    def predict(X):
        tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(tensor)
            probs = torch.sigmoid(logits).cpu().numpy()
        # return probability of LIVE (class 1) for each sample
        return probs.flatten()
    return predict

def explain_mode(mode):
    print(f"\n{'='*60}")
    print(f"  XAI Analysis — Mode: {mode}")
    print(f"{'='*60}")

    # Load trained model for this mode
    model_path = f"models/liveness_{mode}.pth"
    if not os.path.exists(model_path):
        print(f"  [!] No saved model found at {model_path}. Run main_nn.py first.")
        return

    classifier = MLPLivenessClassifier.load_model(model_path)
    model = classifier.model
    device = classifier.device
    predict_fn = make_predict_fn(model, device)

    # Load background samples for SHAP (use training live+spoof mix as baseline)
    # SHAP needs a "background" distribution to compare against
    evaluator = CASIAEvaluator(DATASET_PATH, mode)
    X_train, y_train = evaluator._load_features('train')
    background = X_train[np.random.choice(len(X_train), size=100, replace=False)]

    # Build SHAP explainer
    explainer = shap.KernelExplainer(predict_fn, background)

    # Collect test samples to explain
    live_paths = load_sample_images('test', 'live')
    spoof_paths = load_sample_images('test', 'spoof')
    sample_paths = live_paths + spoof_paths
    sample_labels = ['live'] * len(live_paths) + ['spoof'] * len(spoof_paths)

    os.makedirs(f"xai_output/shap/{mode}", exist_ok=True)

    for img_path, true_label in zip(sample_paths, sample_labels):
        feat = process_cropped_image(img_path, mode)
        if feat is None:
            continue

        # Compute SHAP values for this one sample
        shap_values = explainer.shap_values(feat.reshape(1, -1), nsamples=200)
        shap_arr = shap_values[0]   # shape: (feature_dim,)

        # Get model prediction
        pred_prob = predict_fn(feat.reshape(1, -1))[0]
        pred_label = "live" if pred_prob >= 0.5 else "spoof"

        # ── Visualize ──────────────────────────────────────────────────────
        orig_img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        # For LPQ (histogram, 256-dim) we can't reshape to 64x64, so skip heatmap
        if mode == 'nn_lpq':
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            axes[0].imshow(orig_img)
            axes[0].set_title(f"True: {true_label} | Pred: {pred_label} ({pred_prob:.2f})")
            axes[0].axis('off')

            axes[1].bar(range(len(shap_arr)), shap_arr,
                        color=['red' if v > 0 else 'blue' for v in shap_arr])
            axes[1].set_title("SHAP values (LPQ bins)")
            axes[1].set_xlabel("LPQ bin index")
            axes[1].set_ylabel("SHAP value")

        else:
            # All other modes: reshape 4096 features → 64×64 heatmap
            heatmap = shap_arr.reshape(64, 64)
            heatmap_resized = cv2.resize(heatmap, (orig_img.shape[1], orig_img.shape[0]))

            # Normalize to [-1, 1] for display
            abs_max = np.abs(heatmap_resized).max() + 1e-8
            heatmap_norm = heatmap_resized / abs_max  # range [-1, 1]

            # Convert to color: positive=red (spoof signal), negative=blue (live signal)
            heatmap_color = plt.cm.RdBu_r((heatmap_norm + 1) / 2)[:, :, :3]
            heatmap_color = (heatmap_color * 255).astype(np.uint8)

            # Blend overlay onto original
            overlay = cv2.addWeighted(orig_img, 0.5, heatmap_color, 0.5, 0)

            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(orig_img)
            axes[0].set_title(f"Original\nTrue: {true_label}")
            axes[0].axis('off')

            axes[1].imshow(heatmap_color)
            axes[1].set_title("SHAP Heatmap\nRed=spoof signal, Blue=live signal")
            axes[1].axis('off')

            axes[2].imshow(overlay)
            axes[2].set_title(f"Overlay\nPred: {pred_label} ({pred_prob:.2f})")
            axes[2].axis('off')

        fig.suptitle(f"Mode: {mode} — {os.path.basename(img_path)}", fontsize=12)
        plt.tight_layout()

        stem = os.path.splitext(os.path.basename(img_path))[0]
        out_name = f"xai_output/shap/{mode}/{true_label}_{stem}.png"
        plt.savefig(out_name, dpi=100, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {out_name}")

if __name__ == "__main__":
    for mode in MODES:
        explain_mode(mode)

    print("\nDone. Check the xai_output/ folder for heatmaps.")


/Users/naijawebmaster/Documents/GitHub/context-awareness-and-security-analysis/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  XAI Analysis — Mode: nn_gradients


100%|██████████| 1/1 [00:01<00:00,  1.82s/it]


  Saved: xai_output/shap/nn_gradients/live_s25v2f188.png


100%|██████████| 1/1 [00:01<00:00,  1.42s/it]


...

  Saved: xai_output/shap/nn_gradients/spoof_s8v3f7.png


100%|██████████| 1/1 [00:01<00:00,  1.24s/it]


  Saved: xai_output/shap/nn_gradients/spoof_s21vHR_1f56.png


100%|██████████| 1/1 [00:00<00:00,  1.12it/s]


  Saved: xai_output/shap/nn_gradients/spoof_s29vHR_3f186.png

  XAI Analysis — Mode: nn_raw_gray
  [!] No saved model found at models/liveness_nn_raw_gray.pth. Run main_nn.py first.

  XAI Analysis — Mode: nn_spatial_lbp
  [!] No saved model found at models/liveness_nn_spatial_lbp.pth. Run main_nn.py first.

  XAI Analysis — Mode: nn_high_freq
  [!] No saved model found at models/liveness_nn_high_freq.pth. Run main_nn.py first.

  XAI Analysis — Mode: nn_lpq
  [!] No saved model found at models/liveness_nn_lpq.pth. Run main_nn.py first.

Done. Check the xai_output/ folder for heatmaps.


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import shap
from lime import lime_tabular

from main_nn import process_cropped_image, CASIAEvaluator
from networkb import MLPLivenessClassifier
from xai_gradcam import GradCAMForMLP, _normalize
from xai_scouter import SCOUTERClassifier

DATASET_PATH = "./casia-fasd"
MODES = ['nn_gradients', 'nn_raw_gray', 'nn_spatial_lbp', 'nn_high_freq', 'nn_lpq']
SHAP_BACKGROUND = 100
SHAP_NSAMPLES = 200
LIME_NSAMPLES = 1000
LIME_FEATURES = 50
OUT_DIR = "xai_output/comparison"


def _load_one_image(split, category):
    folder = os.path.join(DATASET_PATH, split, category)
    files = sorted([
        os.path.join(folder, f) for f in os.listdir(folder)
        if f.lower().endswith(('.jpg', '.png', '.bmp'))
    ])
    return files[0] if files else None


def _shap_predict_fn(model, device):
    def fn(X):
        t = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            return torch.sigmoid(model(t)).cpu().numpy().flatten()
    return fn


def _lime_predict_fn(model, device):
    def fn(X):
        t = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            p = torch.sigmoid(model(t)).cpu().numpy().flatten()
        return np.stack([1 - p, p], axis=1)
    return fn


def _colorize_1d(arr, shape2d, cmap='jet'):
    norm = _normalize(arr.reshape(shape2d))
    return (plt.get_cmap(cmap)(norm)[:, :, :3] * 255).astype(np.uint8)


def _overlay(orig, color_map, shape2d):
    h, w = orig.shape[:2]
    resized = cv2.resize(color_map.reshape(shape2d[0], shape2d[1], 3), (w, h))
    return cv2.addWeighted(orig, 0.5, resized, 0.5, 0)


def compare_mode(mode):
    print(f"\n{'='*60}")
    print(f"  XAI Comparison - Mode: {mode}")
    print(f"{'='*60}")

    model_path = f"models/liveness_{mode}.pth"
    if not os.path.exists(model_path):
        print(f"  [!] No model at {model_path}. Run main_nn.py first.")
        return

    classifier = MLPLivenessClassifier.load_model(model_path)
    model = classifier.model
    device = classifier.device

    evaluator = CASIAEvaluator(DATASET_PATH, mode)
    X_train, _ = evaluator._load_features('train')
    feature_dim = X_train.shape[1]
    is_spatial = (feature_dim == 4096)

    shap_predict = _shap_predict_fn(model, device)
    lime_predict = _lime_predict_fn(model, device)

    np.random.seed(42)
    bg_idx = np.random.choice(len(X_train), size=SHAP_BACKGROUND, replace=False)
    shap_explainer = shap.KernelExplainer(shap_predict, X_train[bg_idx])

    lime_explainer = lime_tabular.LimeTabularExplainer(
        training_data=X_train,
        feature_names=[f"f_{i}" for i in range(feature_dim)],
        class_names=['spoof', 'live'],
        mode='classification',
        discretize_continuous=True,
        random_state=42,
    )

    gradcam = GradCAMForMLP(model)

    scouter_path = f"models/scouter_{mode}.pth"
    scouter_clf = None
    if os.path.exists(scouter_path):
        scouter_clf = SCOUTERClassifier.load(scouter_path)
    else:
        print(f"  [!] No SCOUTER model at {scouter_path}. Run xai_scouter.py first.")

    out_dir = os.path.join(OUT_DIR, mode)
    os.makedirs(out_dir, exist_ok=True)

    for category in ('live', 'spoof'):
        img_path = _load_one_image('test', category)
        if img_path is None:
            continue
        feat = process_cropped_image(img_path, mode)
        if feat is None:
            continue

        orig_img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        # Grad-CAM
        x_t = torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(device)
        cam_arr, _, pred_prob = gradcam.compute(x_t)
        pred_label = "live" if pred_prob >= 0.5 else "spoof"

        # SHAP
        shap_vals = shap_explainer.shap_values(feat.reshape(1, -1), nsamples=SHAP_NSAMPLES)
        shap_arr = shap_vals[0]

        # LIME
        exp = lime_explainer.explain_instance(
            feat, lime_predict,
            num_features=LIME_FEATURES, num_samples=LIME_NSAMPLES, labels=(1,)
        )
        lime_arr = np.zeros(feature_dim)
        for idx, w in exp.local_exp[1]:
            lime_arr[idx] = w

        scouter_arr = None
        if scouter_clf is not None:
            _, pos_attn, neg_attn = scouter_clf.explain(feat)
            scouter_arr = pos_attn - neg_attn

        num_panels = 5 if scouter_arr is not None else 4

        if is_spatial:
            shape2d = (64, 64)
            cam_color   = _colorize_1d(np.abs(cam_arr),  shape2d, cmap='jet')
            shap_color  = _colorize_1d(np.abs(shap_arr), shape2d, cmap='RdBu_r')
            lime_color  = _colorize_1d(np.abs(lime_arr), shape2d, cmap='PiYG')

            h, w = orig_img.shape[:2]
            def blend(c): return cv2.addWeighted(orig_img, 0.5, cv2.resize(c, (w, h)), 0.5, 0)

            fig, axes = plt.subplots(1, num_panels, figsize=(5 * num_panels, 4))
            for ax in axes:
                ax.axis('off')
            axes[0].imshow(orig_img)
            axes[0].set_title(f"Original\nTrue: {category} | Pred: {pred_label} ({pred_prob:.2f})")
            axes[1].imshow(blend(cam_color))
            axes[1].set_title("Grad-CAM\n(gradient x activation, projected to input)")
            axes[2].imshow(blend(shap_color))
            axes[2].set_title("SHAP\n(Shapley attribution vs background)")
            axes[3].imshow(blend(lime_color))
            axes[3].set_title("LIME\n(local linear surrogate weights)")
            if scouter_arr is not None:
                scouter_norm = _normalize(scouter_arr.reshape(64, 64))
                scouter_color = _colorize_1d(scouter_norm, shape2d, cmap='RdYlGn')
                axes[4].imshow(blend(scouter_color))
                axes[4].set_title("SCOUTER\n(slot attention: green=live, red=spoof)")

        else:
            fig, axes = plt.subplots(1, num_panels, figsize=(5 * num_panels, 4))
            axes[0].imshow(orig_img); axes[0].axis('off')
            axes[0].set_title(f"Original\nTrue: {category} | Pred: {pred_label} ({pred_prob:.2f})")

            axes[1].bar(range(len(cam_arr)), np.abs(cam_arr), color='steelblue')
            axes[1].set_title("Grad-CAM importance"); axes[1].set_xlabel("LPQ bin")

            axes[2].bar(range(len(shap_arr)), shap_arr,
                        color=['red' if v > 0 else 'blue' for v in shap_arr])
            axes[2].set_title("SHAP values\nRed=spoof signal, Blue=live signal")
            axes[2].set_xlabel("LPQ bin")

            axes[3].bar(range(len(lime_arr)), lime_arr,
                        color=['green' if v > 0 else 'purple' for v in lime_arr])
            axes[3].set_title("LIME weights\nGreen=live signal, Purple=spoof signal")
            axes[3].set_xlabel("LPQ bin")

            if scouter_arr is not None:
                axes[4].bar(range(len(scouter_arr)), scouter_arr,
                            color=['green' if v > 0 else 'red' for v in scouter_arr])
                axes[4].set_title("SCOUTER net evidence\nGreen=live, Red=spoof")
                axes[4].set_xlabel("LPQ bin")

        fig.suptitle(f"XAI Comparison -- Mode: {mode} | {category}", fontsize=12)
        plt.tight_layout()
        out_path = f"{out_dir}/{category}_comparison.png"
        plt.savefig(out_path, dpi=100, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {out_path}")

    gradcam.remove_hooks()


ANALYSIS_REPORT = """
╔══════════════════════════════════════════════════════════════════╗
║         XAI METHOD COMPARATIVE ANALYSIS                         ║
║         Liveness Detection -- CASIA-FASD Dataset                ║
╚══════════════════════════════════════════════════════════════════╝

Four explainability methods were applied to the trained LivenessNet
MLP classifier across all five feature extraction modes (nn_gradients,
nn_raw_gray, nn_spatial_lbp, nn_high_freq, nn_lpq).

  Recommended workflow:
    Train model -> Grad-CAM (rapid screening of all test samples)
    -> SHAP (deep per-sample analysis for report figures)
    -> LIME (sanity-check agreement with SHAP on key samples)
    -> SCOUTER (intrinsic baseline for spatial feature modes)
"""


def save_report():
    os.makedirs(OUT_DIR, exist_ok=True)
    report_path = os.path.join(OUT_DIR, "xai_comparative_analysis.txt")
    with open(report_path, "w") as f:
        f.write(ANALYSIS_REPORT)
    print(ANALYSIS_REPORT)
    print(f"  Report saved -> {report_path}")


if __name__ == "__main__":
    for mode in MODES:
        compare_mode(mode)
    save_report()
    print("\nDone. Check xai_output/comparison/ for figures and report.")